In [ ]:
def scene_callback(self, msg):
        try:
            data = json.loads(msg.data)
            objects = data.get('objects', [])

            if not objects:
                return

            self.perceptual_graph.clear()

            unique_concepts = set()
            new_concepts = set()

            for obj in objects:
                object_type = obj['label']
                confidence  = obj.get('conf', 1.0)

                pos = obj['position']
                x, y, z = pos['x'], pos['y'], pos['z']

                position = (x, y, z)

                if object_type not in self.knowledge_base.concepts:
                    self.knowledge_base.learn_concept(object_type)
                    new_concepts.add(object_type)

                unique_concepts.add(object_type)

                self.perceptual_graph.add_instance(
                    concept=object_type,
                    position=position,
                    confidence=confidence,
                    bbox=None 
                )

            self.perceptual_graph.compute_spatial_relations()

            # Actualizar grafo semántico
            if new_concepts:
                self.semantic_graph.build_full_action_graph(
                    new_concepts,
                    self.robot_capabilities.actions
                )

            self.run_planning_pipeline()

        except Exception as e:
            self.get_logger().error(f"Scene callback error: {e}")

def run_planning_pipeline(self):
    if not hasattr(self, "current_goal") or self.current_goal is None:
        if self.mode == 'auto':
            self.get_logger().info("Generando objetivo inicial con LLM...")

            self.candidate_goals = self.goal_generator.generate_goals(
                perceptual_graph=self.perceptual_graph,
                semantic_graph=self.semantic_graph.graph,
                use_llm=True,
                mode='auto',
                user_text=None
            )

            if self.candidate_goals:
                # Escoger el de mayor prioridad
                chosen_goal = max(self.candidate_goals, key=lambda g: g.priority)
                self.current_goal = chosen_goal

                self.get_logger().info("Objetivos sugeridos:")
                for g in self.candidate_goals:
                    self.get_logger().info(f" - {g.goal_text} (priority={g.priority})")

                self.get_logger().info(f"Objetivo seleccionado: {self.current_goal.goal_text}")
            else:
                self.get_logger().error("No se generaron objetivos.")
        if self.mode == 'manual':
            if self.user_goal is None:
                return          
            if self.conversation_started:
                return

            self.conversation_started = True              

            self.candidate_goals = self.goal_generator.generate_goals(
                perceptual_graph=self.perceptual_graph,
                semantic_graph=self.semantic_graph.graph,
                use_llm=True,
                mode='manual',
                user_text=self.user_goal
            )
            self.get_logger().info(f"Objetivos generados: {len(self.candidate_goals) if self.candidate_goals else 0}")

            if self.candidate_goals:
                # Escoger el de mayor prioridad
                chosen_goal = max(self.candidate_goals, key=lambda g: g.priority)
                self.current_goal = chosen_goal

                self.get_logger().info("Objetivos sugeridos:")
                for g in self.candidate_goals:
                    self.get_logger().info(f" - {g.goal_text} (priority={g.priority})")

                self.get_logger().info(f"Objetivo seleccionado: {self.current_goal.goal_text}")
            else:
                self.get_logger().error("No se generaron objetivos.")
            
            # Recorrer usando meta-edge-task-dijkstra
    if self.current_goal and self.plan_logged == False:
        target_objects = self.current_goal.target_objects
        required_actions = self.current_goal.required_actions

        # Extraer targets si están vacíos
        if not target_objects:
            self.get_logger().warn("No target objects specified, extracting from goal text")
            target_objects = []
            for concept_id in self.semantic_graph.concept_nodes.keys():
                if concept_id.split("_")[0] in self.current_goal.goal_text.lower():
                    target_objects.append(concept_id)
        
        if not target_objects:
            self.get_logger().warn("No se encontraron target objects.")
            return
        
        # Extraer acciones si están vacías
        if not required_actions:
            self.get_logger().warn("No required actions specified, extracting from goal")
            required_actions = []
            for action_id in self.robot_capabilities.actions.keys():
                if action_id in self.current_goal.goal_text.lower():
                    required_actions.append(action_id)
        
        if not required_actions:
            self.get_logger().warn("No se encontraron required actions.")
            return
        
        self.get_logger().info(f"Goal: {self.current_goal.goal_text}")
        self.get_logger().info(f"Target objects: {target_objects}")
        self.get_logger().info(f"Required actions: {self.current_goal.required_actions}")
        
        full_plan_steps = []

        # Posibles nodos iniciales del robot 
        possible_starts = [
            node_id for node_id in self.semantic_graph.concept_nodes.keys()
            if node_id.startswith("robot_")
        ]
        
        if not possible_starts:
            self.get_logger().warn("No hay nodos de inicio válidos para el robot.")
            return
        
        # Generar plan usando m-etd con fallback
        success, full_plan, selected_goal_text = self.execute_goal_with_fallback(
            candidate_goals=self.candidate_goals,
            start_nodes=possible_starts
        )
        
        if not success:
            self.get_logger().error("═══════════════════════════════════════════")
            self.get_logger().error("NO OBJECTIVES LEFT")
            self.get_logger().error("No se pudo generar plan para ningún objetivo")
            self.get_logger().error("═══════════════════════════════════════════")
            return

        self.get_logger().info(f"Objetivo seleccionado: {selected_goal_text}")
        
        # Publicar plan completo
        
        plan_string = f"{full_plan[0][0]} → {full_plan[0][1]} → {full_plan[0][2]}"
        publish_action = f"{full_plan[0][1]}"
        for (u, action, v) in full_plan[1:]:
            plan_string += f" then {u} → {action} → {v}"
            publish_action += f" {action}"

        self.full_plan = full_plan

        self.get_logger().info(f"═══════════════════════════════════════════")
        self.get_logger().info(f"PLAN COMPLETO GENERADO")
        self.get_logger().info(f"Goal: {selected_goal_text}")
        self.get_logger().info(f"═══════════════════════════════════════════")
        self.get_logger().info(plan_string)
        self.get_logger().info(f"═══════════════════════════════════════════")
        
        # Publicar plan
        plan_msg = String()
        plan_msg.data = publish_action
        self.results_pub.publish(plan_msg)
        
        self.plan_logged = True

        if self.mode == 'manual':
            self.current_goal = None
            self.user_goal = None
            self.goal_processed = False
            self.get_logger().info("Esperando nuevo objetivo.")


    # Log
    if not hasattr(self, "_last_scene_snapshot"):
        self._last_scene_snapshot = (0, 0, 0, 0)  # (num_instances, num_relations, num_concepts, num_actions)

    # Obtener estadísticas actuales
    perceptual_stats = self.perceptual_graph.to_json()["statistics"]
    semantic_stats = self.semantic_graph.export_to_json()["statistics"]
    current_snapshot = (
        perceptual_stats["total_instances"],
        perceptual_stats["total_relations"],
        semantic_stats["total_concepts"],
        semantic_stats["total_actions"]
    )

    # Log solo si cambió
    if current_snapshot != self._last_scene_snapshot:
        self.get_logger().info(
            f"Scene Update:\n"
            f"  Perceptual: {perceptual_stats['total_instances']} instances, "
            f"{perceptual_stats['total_relations']} spatial relations\n"
            f"  Semantic: {semantic_stats['total_concepts']} concepts, "
            f"{semantic_stats['total_actions']} possible actions"
        )
        self._last_scene_snapshot = current_snapshot

    
    # Mostrar relaciones detectadas
    self._log_spatial_relations()

In [ ]:
def yolo_callback(self, msg):
        """Reconstruye grafo perceptual y actualiza semántico"""
        try:
            data = json.loads(msg.data)
            detections = data.get('detections', [])
            
            if not detections:
                return
            
            # Limpiar grafo geométrico
            self.perceptual_graph.clear()
            
            # Extraer conceptos únicos
            unique_concepts = set()
            new_concepts = set()
            
            # Añadir instancias
            for det in detections:
                object_type = det['class_name']
                confidence = det['confidence']

                center = det.get('center', [0, 0])
                bbox = det.get('bbox', [center[0]-25, center[1]-25, 50, 50])
                
                x = bbox[0] + bbox[2] / 2
                y = bbox[1] + bbox[3] / 2
                z = self.get_depth_from_bbox(bbox)
            
                if z > 0:
                    self.get_logger().debug(
                        f"{object_type} at ({x:.1f}, {y:.1f}, {z:.3f}m)"
                    )
                
                position = (x, y, z)

                # Aprender nuevo concepto si es necesario
                if object_type not in self.knowledge_base.concepts:
                    self.knowledge_base.learn_concept(object_type)
                    new_concepts.add(object_type)
                
                unique_concepts.add(object_type)

                # Añadir al grafo perceptual
                self.perceptual_graph.add_instance(
                    concept=object_type,
                    position=position,
                    confidence=confidence,
                    bbox=tuple(bbox)
                )
            
            # Calcular relaciones espaciales
            self.perceptual_graph.compute_spatial_relations()

            # Actualizar grafo semántico si hay nuevos conceptos
            if new_concepts:
                self.semantic_graph.build_full_action_graph(
                    new_concepts,
                    self.robot_capabilities.actions
                )
            # Generar objetivo
            if not hasattr(self, "current_goal") or self.current_goal is None:
                if self.mode == 'auto':
                    self.get_logger().info("Generando objetivo inicial con LLM...")

                    self.candidate_goals = self.goal_generator.generate_goals(
                        perceptual_graph=self.perceptual_graph,
                        semantic_graph=self.semantic_graph.graph,
                        use_llm=True,
                        mode='auto',
                        user_text=None
                    )

                    if self.candidate_goals:
                        # Escoger el de mayor prioridad
                        chosen_goal = max(self.candidate_goals, key=lambda g: g.priority)
                        self.current_goal = chosen_goal

                        self.get_logger().info("Objetivos sugeridos:")
                        for g in self.candidate_goals:
                            self.get_logger().info(f" - {g.goal_text} (priority={g.priority})")

                        self.get_logger().info(f"Objetivo seleccionado: {self.current_goal.goal_text}")
                    else:
                        self.get_logger().error("No se generaron objetivos.")

                if self.mode == 'manual':
                    if self.user_goal is None:
                        return          
                    if self.conversation_started:
                        return

                    self.conversation_started = True              

                    self.candidate_goals = self.goal_generator.generate_goals(
                        perceptual_graph=self.perceptual_graph,
                        semantic_graph=self.semantic_graph.graph,
                        use_llm=True,
                        mode='manual',
                        user_text=self.user_goal
                    )
                    self.get_logger().info(f"Objetivos generados: {len(self.candidate_goals) if self.candidate_goals else 0}")

                    if self.candidate_goals:
                        # Escoger el de mayor prioridad
                        chosen_goal = max(self.candidate_goals, key=lambda g: g.priority)
                        self.current_goal = chosen_goal

                        self.get_logger().info("Objetivos sugeridos:")
                        for g in self.candidate_goals:
                            self.get_logger().info(f" - {g.goal_text} (priority={g.priority})")

                        self.get_logger().info(f"Objetivo seleccionado: {self.current_goal.goal_text}")
                    else:
                        self.get_logger().error("No se generaron objetivos.")
            
            # Recorrer usando meta-edge-task-dijkstra
            if self.current_goal and self.plan_logged == False:
                target_objects = self.current_goal.target_objects
                required_actions = self.current_goal.required_actions

                # Extraer targets si están vacíos
                if not target_objects:
                    self.get_logger().warn("No target objects specified, extracting from goal text")
                    target_objects = []
                    for concept_id in self.semantic_graph.concept_nodes.keys():
                        if concept_id.split("_")[0] in self.current_goal.goal_text.lower():
                            target_objects.append(concept_id)
                
                if not target_objects:
                    self.get_logger().warn("No se encontraron target objects.")
                    return
                
                # Extraer acciones si están vacías
                if not required_actions:
                    self.get_logger().warn("No required actions specified, extracting from goal")
                    required_actions = []
                    for action_id in self.robot_capabilities.actions.keys():
                        if action_id in self.current_goal.goal_text.lower():
                            required_actions.append(action_id)
                
                if not required_actions:
                    self.get_logger().warn("No se encontraron required actions.")
                    return
                
                self.get_logger().info(f"Goal: {self.current_goal.goal_text}")
                self.get_logger().info(f"Target objects: {target_objects}")
                self.get_logger().info(f"Required actions: {self.current_goal.required_actions}")
                
                full_plan_steps = []

                # Posibles nodos iniciales del robot 
                possible_starts = [
                    node_id for node_id in self.semantic_graph.concept_nodes.keys()
                    if node_id.startswith("robot_")
                ]
                
                if not possible_starts:
                    self.get_logger().warn("No hay nodos de inicio válidos para el robot.")
                    return
                
                # Generar plan usando m-etd con fallback
                success, full_plan, selected_goal_text = self.execute_goal_with_fallback(
                    candidate_goals=self.candidate_goals,
                    start_nodes=possible_starts
                )
                
                if not success:
                    self.get_logger().error("═══════════════════════════════════════════")
                    self.get_logger().error("NO OBJECTIVES LEFT")
                    self.get_logger().error("No se pudo generar plan para ningún objetivo")
                    self.get_logger().error("═══════════════════════════════════════════")
                    return

                self.get_logger().info(f"Objetivo seleccionado: {selected_goal_text}")
                
                # Publicar plan completo
                
                plan_string = f"{full_plan[0][0]} → {full_plan[0][1]} → {full_plan[0][2]}"
                publish_action = f"{full_plan[0][1]}"
                for (u, action, v) in full_plan[1:]:
                    plan_string += f" then {u} → {action} → {v}"
                    publish_action += f" {action}"
        
                self.full_plan = full_plan

                self.get_logger().info(f"═══════════════════════════════════════════")
                self.get_logger().info(f"PLAN COMPLETO GENERADO")
                self.get_logger().info(f"Goal: {selected_goal_text}")
                self.get_logger().info(f"═══════════════════════════════════════════")
                self.get_logger().info(plan_string)
                self.get_logger().info(f"═══════════════════════════════════════════")
                
                # Publicar plan
                plan_msg = String()
                plan_msg.data = publish_action
                self.results_pub.publish(plan_msg)
                
                self.plan_logged = True

                if self.mode == 'manual':
                    self.current_goal = None
                    self.user_goal = None
                    self.goal_processed = False
                    self.get_logger().info("Esperando nuevo objetivo.")


            # Log
            if not hasattr(self, "_last_scene_snapshot"):
                self._last_scene_snapshot = (0, 0, 0, 0)  # (num_instances, num_relations, num_concepts, num_actions)

            # Obtener estadísticas actuales
            perceptual_stats = self.perceptual_graph.to_json()["statistics"]
            semantic_stats = self.semantic_graph.export_to_json()["statistics"]
            current_snapshot = (
                perceptual_stats["total_instances"],
                perceptual_stats["total_relations"],
                semantic_stats["total_concepts"],
                semantic_stats["total_actions"]
            )

            # Log solo si cambió
            if current_snapshot != self._last_scene_snapshot:
                self.get_logger().info(
                    f"Scene Update:\n"
                    f"  Perceptual: {perceptual_stats['total_instances']} instances, "
                    f"{perceptual_stats['total_relations']} spatial relations\n"
                    f"  Semantic: {semantic_stats['total_concepts']} concepts, "
                    f"{semantic_stats['total_actions']} possible actions"
                )
                self._last_scene_snapshot = current_snapshot

            
            # Mostrar relaciones detectadas
            self._log_spatial_relations()
            
        except Exception as e:
            self.get_logger().error(f"YOLO callback error: {e}")